# Taller Pandas: 120 años de historia olímpica

**Curso:** Programación II, Ciencia de Datos
**Grupo:** taller-pandas-olimpiadas-grupoX

En este notebook analizamos los datos de Kaggle *120 years of Olympic history: athletes and results* (`athlete_events.csv` y `noc_regions.csv`). Seguimos el orden de los 14 puntos del taller: carga, descripción, nuevas variables, fusión, datos faltantes, preguntas con `groupby` y tablas de contingencia, gráficos y conclusiones.

> Cada fila de `athlete_events.csv` es **un atleta en una prueba de una edición de los Juegos**, no un atleta único. Lo tenemos en cuenta en todo el análisis.

Los archivos originales deben estar en la carpeta `data/` (ver `data/README.md`). Las tablas se guardan en `outputs/tablas/` y los gráficos en `outputs/graficos/`.

In [1]:
# Librerías y rutas del proyecto
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

DATA = "../data"
TABLAS = "../outputs/tablas"
GRAFICOS = "../outputs/graficos"
os.makedirs(TABLAS, exist_ok=True)
os.makedirs(GRAFICOS, exist_ok=True)

## 1. Carga de los datos

Cargamos cada archivo en un dataframe distinto: `atletas` y `regiones`.

In [2]:
# Punto 1: carga de cada dataset en un dataframe distinto
atletas = pd.read_csv(f"{DATA}/athlete_events.csv")
regiones = pd.read_csv(f"{DATA}/noc_regions.csv")

print("atletas:", atletas.shape)
print("regiones:", regiones.shape)
atletas.head()

atletas: (271116, 15)
regiones: (230, 3)


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [3]:
regiones.head()

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


`atletas` tiene **271.116 filas y 15 columnas**, y `regiones` tiene **230 filas y 3 columnas**.

## 2. Descripción de las variables

**athlete_events.csv** (`atletas`)

| Variable | Tipo | Descripción |
|---|---|---|
| ID | entero | Identificador único de cada atleta |
| Name | texto | Nombre del atleta |
| Sex | categórica | Sexo: M (hombre) o F (mujer) |
| Age | numérica | Edad en años al momento de la prueba |
| Height | numérica | Estatura en centímetros |
| Weight | numérica | Peso en kilogramos |
| Team | texto | Nombre del equipo con el que compitió |
| NOC | categórica | Código de tres letras del Comité Olímpico Nacional |
| Games | categórica | Edición de los Juegos (año + temporada) |
| Year | entero | Año de la edición |
| Season | categórica | Temporada: Summer (verano) o Winter (invierno) |
| City | texto | Ciudad sede |
| Sport | categórica | Deporte |
| Event | categórica | Prueba específica dentro del deporte |
| Medal | categórica | Gold, Silver, Bronze o vacío (NA) si no ganó |

**noc_regions.csv** (`regiones`)

| Variable | Tipo | Descripción |
|---|---|---|
| NOC | categórica | Código del Comité Olímpico Nacional (llave para unir) |
| region | texto | País o región a la que corresponde el NOC |
| notes | texto | Notas aclaratorias (por ejemplo, nombres históricos como Bohemia o Yugoslavia) |

In [4]:
# Punto 2: tipos de dato y valores no nulos de cada variable
atletas.info()
print()
regiones.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 31.0 MB

<class 'pandas.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   NOC     230 non-null    s

Las variables numéricas son `ID`, `Age`, `Height`, `Weight` y `Year` (aunque `ID` y `Year` funcionan más como identificadores que como medidas). Todo lo demás es texto o categoría. Desde ya se ve que `Age`, `Height`, `Weight` y `Medal` tienen menos valores no nulos que el total de filas, y en `regiones` faltan algunos valores de `region` (3) y casi todas las `notes`.

## 3. Estadísticas básicas y comparación entre dataframes

In [5]:
# Punto 3: estadísticas básicas de las variables numéricas de atletas
atletas.describe().round(2)

,ID,Age,Height,Weight,Year
count,271116.00,261642.00,210945.00,208241.00,271116.00
mean,68248.95,25.56,175.34,70.70,1978.38
std,39022.29,6.39,10.52,14.35,29.88
min,1.00,10.00,127.00,25.00,1896.00
25%,34643.00,21.00,168.00,60.00,1960.00
50%,68205.00,24.00,175.00,70.00,1988.00
75%,102097.25,28.00,183.00,79.00,2002.00
max,135571.00,97.00,226.00,214.00,2016.00


In [6]:
# Estadísticas de las variables categóricas de atletas
atletas.describe(exclude="number")

,Name,Sex,Team,NOC,Games,Season,City,Sport,Event,Medal
count,271116,271116,271116,271116,271116,271116,271116,271116,271116,39783
unique,134732,2,1184,230,51,2,42,66,765,3
top,Robert Tait McKenzie,M,United States,USA,2000 Summer,Summer,London,Athletics,Football Men's Football,Gold
freq,58,196594,17847,18853,13821,222552,22426,38624,5733,13372


In [7]:
# Estadísticas de regiones
regiones.describe(include="all")

,NOC,region,notes
count,230,227,21
unique,230,206,21
top,AFG,Germany,Netherlands Antilles
freq,1,4,1


In [8]:
# Comparación entre dataframes a través de la llave NOC
noc_atletas = set(atletas["NOC"].unique())
noc_regiones = set(regiones["NOC"].unique())

print("NOC distintos en atletas:", len(noc_atletas))
print("NOC distintos en regiones:", len(noc_regiones))
print("NOC en atletas que no están en regiones:", sorted(noc_atletas - noc_regiones))
print("NOC en regiones que no están en atletas:", sorted(noc_regiones - noc_atletas))
print("Regiones distintas:", regiones["region"].nunique())
print("Equipos (Team) distintos en atletas:", atletas["Team"].nunique())

NOC distintos en atletas: 230
NOC distintos en regiones: 230
NOC en atletas que no están en regiones: ['SGP']
NOC en regiones que no están en atletas: ['SIN']
Regiones distintas: 206
Equipos (Team) distintos en atletas: 1184


**Interpretación**

- La edad promedio por registro es de **25,6 años** (mediana 24), con un rango muy amplio: de 10 a 97 años.
- La estatura promedio es de **175,3 cm** y el peso promedio de **70,7 kg**. El 50 % central de los atletas mide entre 168 y 183 cm.
- Los años van de **1896 a 2016**, pero la mediana es 1988, lo que significa que la mitad de los registros son de los últimos ~30 años.
- Hay **134.732 nombres distintos**, 1.184 equipos, 230 NOC, 51 ediciones, 66 deportes y 765 pruebas. El 72,5 % de los registros son hombres (196.594 de 271.116).
- **Comparación:** ambos dataframes tienen 230 NOC, pero no son exactamente los mismos: `SGP` (Singapur) está en atletas y no en regiones, y `SIN` está en regiones y no en atletas. Además, los 230 NOC se agrupan en solo **206 regiones**, porque varios códigos históricos apuntan al mismo país (por ejemplo, GER, FRG, GDR y SAA corresponden a Germany). `atletas` tiene mucho más detalle (1.184 valores de `Team`) que `regiones`, que solo traduce códigos.

## 4. Variable IMC

$$IMC = \frac{Weight}{(Height/100)^2}$$

In [9]:
# Punto 4: variable IMC = peso / (estatura en metros)^2
atletas["IMC"] = atletas["Weight"] / (atletas["Height"] / 100) ** 2
atletas["IMC"].describe().round(2)

count    206853.00
mean         22.78
std           2.91
min           8.36
25%          20.96
50%          22.53
75%          24.21
max          63.90
Name: IMC, dtype: float64

In [10]:
# IMC promedio por deporte (deportes con al menos 500 registros con IMC)
imc_deporte = (atletas.groupby("Sport")["IMC"]
               .agg(["count", "mean", "std", "min", "max"])
               .query("count >= 500")
               .sort_values("mean")
               .round(2))
imc_deporte.to_csv(f"{TABLAS}/imc_por_deporte.csv")
pd.concat([imc_deporte.head(5), imc_deporte.tail(5)])

,count,mean,std,min,max
Sport,,,,,
Rhythmic Gymnastics,615,17.29,1.31,13.33,22.04
Synchronized Swimming,849,19.67,1.42,13.49,37.29
Triathlon,526,20.39,1.32,17.21,25.28
Figure Skating,1512,20.84,2.15,15.06,28.88
Ski Jumping,1968,20.90,1.92,17.19,28.44
Ice Hockey,4579,25.14,1.94,17.69,33.52
Judo,3384,25.59,4.69,15.15,63.90
Baseball,846,25.68,2.29,17.09,33.77
Bobsleigh,2206,26.99,2.53,17.96,39.54


**¿Qué opinamos del IMC?**

El IMC promedio es de **22,8** y la mitad de los registros está entre 21,0 y 24,2, es decir, en el rango que normalmente se considera "peso normal". Aun así, el IMC no tiene el mismo sentido en todos los deportes:

- Es una medida muy simple, porque no distingue entre masa muscular y grasa. Un levantador de pesas (IMC promedio de **27,7**) o un judoca (**25,6**) tiene un IMC alto por músculo, no por sobrepeso.
- En deportes estéticos o de resistencia es bajo: gimnasia rítmica **17,3**, natación sincronizada **19,7** y triatlón **20,4**.
- En deportes con categorías de peso (judo, halterofilia, lucha) la desviación es alta (4,7 y 5,5) porque se mezclan pesos muy distintos. Por eso el máximo de **63,9** aparece en judo.
- En deportes como tiro, vela o las competencias de arte, el físico casi no influye en el resultado, así que el IMC dice poco.
- Además, solo **206.853** registros tienen IMC, porque falta la estatura o el peso en cerca del 24 % de los casos.

Conclusión: el IMC sirve para describir el tipo de cuerpo de cada deporte, pero no como indicador de salud y no es comparable entre todos los deportes.

## 5. Fusión (merge) con regiones

Usamos `how="left"` porque queremos **conservar todas las filas de atletas** aunque su NOC no aparezca en `regiones`. Un `inner` habría eliminado esas filas y un `outer` habría agregado filas de regiones sin atletas (como `SIN`).

In [11]:
# Punto 5: fusión con how='left' para conservar todas las filas de atletas
filas_antes = len(atletas)
df = atletas.merge(regiones, on="NOC", how="left")
filas_despues = len(df)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("¿Se conservan las filas?", filas_antes == filas_despues)
print("Filas sin región:", df["region"].isna().sum())

Filas antes: 271116
Filas después: 271116
¿Se conservan las filas? True
Filas sin región: 370


In [12]:
# NOC de las filas que quedaron sin región
df[df["region"].isna()].groupby(["NOC", "notes"], dropna=False).size().reset_index(name="filas")

,NOC,notes,filas
0,ROT,Refugee Olympic Team,12
1,SGP,NaN,349
2,TUV,Tuvalu,7
3,UNK,Unknown,2


El número de filas **no cambia** (271.116 antes y después), así que el merge no duplicó filas; esto confirma que `NOC` es única en `regiones`. Quedaron **370 filas sin región**. ¿Por qué ocurre?

- **SGP (349 filas):** Singapur está codificado como `SGP` en atletas y como `SIN` en regiones. Es un error de consistencia entre las dos fuentes, así que lo corregimos abajo.
- **ROT (12), TUV (7) y UNK (2):** el código sí existe en `regiones`, pero su `region` está vacía a propósito, porque son el *Refugee Olympic Team*, Tuvalu y un equipo *Unknown*. No corresponden a un país del listado.

In [13]:
# Corrección: Singapur aparece como SGP en atletas y como SIN en regiones
df.loc[df["NOC"] == "SGP", "region"] = "Singapore"
print("Filas sin región después de corregir SGP:", df["region"].isna().sum())

Filas sin región después de corregir SGP: 21


Después de corregir Singapur, solo quedan **21 filas sin región** (ROT, TUV y UNK).

## 6. Variable binaria `medalla`

Los vacíos de `Medal` **no son datos perdidos**: significan que el atleta no ganó medalla en esa prueba. Por eso los reemplazamos por `"Sin medalla"` y creamos `medalla` = 1 si ganó oro, plata o bronce, y 0 en otro caso.

In [14]:
# Punto 6: variable binaria medalla (los vacíos de Medal significan que no ganó)
df["Medal"] = df["Medal"].fillna("Sin medalla")
df["medalla"] = (df["Medal"] != "Sin medalla").astype(int)
df["Medal"].value_counts()

Medal
Sin medalla    231333
Gold            13372
Bronze          13295
Silver          13116
Name: count, dtype: int64

In [15]:
# Filas frente a atletas únicos
print("Filas (atleta-prueba-edición):", len(df))
print("Atletas únicos (ID):", df["ID"].nunique())
print("Promedio de filas por atleta:", round(len(df) / df["ID"].nunique(), 2))

Filas (atleta-prueba-edición): 271116


Atletas únicos (ID): 135571
Promedio de filas por atleta: 2.0


In [16]:
# Efecto de los deportes de equipo en el conteo de medallas
medallas_filas = df["medalla"].sum()
medallas_reales = (df[df["medalla"] == 1]
                   .drop_duplicates(["Games", "Event", "Medal"])
                   .shape[0])
print("Medallas contando filas:", medallas_filas)
print("Medallas contando una por (edición, prueba, tipo):", medallas_reales)

ejemplo = df[(df["Event"] == "Basketball Men's Basketball") & (df["Games"] == "2016 Summer") & (df["Medal"] == "Gold")]
print("Filas de oro en baloncesto masculino 2016:", len(ejemplo), "-> pero es una sola medalla para", list(ejemplo["Team"].unique()))

Medallas contando filas: 39783
Medallas contando una por (edición, prueba, tipo): 18293
Filas de oro en baloncesto masculino 2016: 12 -> pero es una sola medalla para ['United States']


Hay **39.783 registros con medalla** (13.372 oros, 13.116 platas y 13.295 bronces) y 231.333 sin medalla.

Pero el dataframe tiene **271.116 filas** y solo **135.571 atletas únicos**, es decir, en promedio cada atleta aparece 2 veces (porque compite en varias pruebas o en varias ediciones). Esto tiene dos implicaciones al contar medallas:

1. Si contamos filas, contamos **medallas por atleta**, no medallas entregadas. En los deportes de equipo cada integrante tiene su propia fila: el oro de baloncesto masculino 2016 aparece **12 veces** aunque Estados Unidos recibió una sola medalla.
2. Si contamos una medalla por combinación (edición, prueba, tipo de medalla), el total baja de 39.783 a **18.293**, que se acerca más a la cantidad de medallas entregadas como en un medallero oficial (no es exacto, porque hay pruebas con dos bronces o empates).

Por eso, en el resto del taller hablamos de **"registros con medalla"** y comparamos **tasas** (medallas / registros), que no se distorsionan tanto por el tamaño de los equipos.

## 7. Estadísticas de nuevo y representatividad de los datos

In [17]:
# Punto 7: estadísticas nuevamente con las variables nuevas
df[["Age", "Height", "Weight", "Year", "IMC", "medalla"]].describe().round(2)

,Age,Height,Weight,Year,IMC,medalla
count,261642.00,210945.00,208241.00,271116.00,206853.00,271116.00
mean,25.56,175.34,70.70,1978.38,22.78,0.15
std,6.39,10.52,14.35,29.88,2.91,0.35
min,10.00,127.00,25.00,1896.00,8.36,0.00
25%,21.00,168.00,60.00,1960.00,20.96,0.00
50%,24.00,175.00,70.00,1988.00,22.53,0.00
75%,28.00,183.00,79.00,2002.00,24.21,0.00
max,97.00,226.00,214.00,2016.00,63.90,1.00


In [18]:
# Participación por época (década), temporada y sexo
df["decada"] = (df["Year"] // 10) * 10
part_decada = pd.crosstab(df["decada"], df["Sex"])
part_decada["Total"] = part_decada.sum(axis=1)
part_decada["% mujeres"] = (100 * part_decada["F"] / part_decada["Total"]).round(1)
part_decada.to_csv(f"{TABLAS}/participacion_por_decada_sexo.csv")
part_decada

Sex,F,M,Total,% mujeres
decada,,,,
1890,0,380,380,0.0
1900,107,7964,8071,1.3
1910,87,3953,4040,2.2
1920,832,14727,15559,5.3
1930,918,9804,10722,8.6
1940,761,6719,7480,10.2
1950,2821,12971,15792,17.9
1960,5675,23519,29194,19.4
1970,5214,17247,22461,23.2


In [19]:
# Participación por temporada y sexo
part_temp = pd.crosstab(df["Season"], df["Sex"], margins=True, margins_name="Total")
print(part_temp)
print()
print((100 * pd.crosstab(df["Season"], df["Sex"], normalize="index")).round(1))
print()
print("Ediciones por temporada:")
print(df.groupby("Season")["Games"].nunique())

Sex         F       M   Total
Season                       
Summer  59443  163109  222552
Winter  15079   33485   48564
Total   74522  196594  271116

Sex        F     M
Season            
Summer  26.7  73.3
Winter  31.0  69.0

Ediciones por temporada:
Season
Summer    29
Winter    22
Name: Games, dtype: int64


**¿Son los datos representativos de toda la historia?** No de forma pareja:

- **Por época:** la participación crece muchísimo con el tiempo. La década de 1890 tiene 380 registros y la de 2000 tiene 49.357. La mediana del año es 1988, así que la información está cargada hacia los Juegos modernos. Las décadas de 1910 y 1940 tienen pocos registros por las guerras mundiales (no hubo Juegos en 1916, 1940 ni 1944).
- **Por sexo:** el porcentaje de mujeres pasa de **0 %** en 1896 a **1,3 %** en 1900, **17,9 %** en los 50 y **44,3 %** en la década de 2010. Cualquier conclusión sobre mujeres está dominada por los Juegos recientes.
- **Por temporada:** el verano tiene **222.552** registros (82 %) frente a **48.564** del invierno, y además 29 ediciones contra 22. Los Juegos de Invierno empezaron en 1924 y son más pequeños. La proporción de mujeres es un poco mayor en invierno (31,0 % vs 26,7 %).
- La media de `medalla` es 0,15: aproximadamente el **15 %** de los registros terminó con medalla.

En resumen, los datos sí cubren 120 años, pero la mayor parte de la información (y la más completa) es de la segunda mitad del siglo XX en adelante, de los Juegos de Verano y de hombres.

## 8. Datos faltantes

In [20]:
# Punto 8: datos faltantes por variable
conteo = df.isna().sum()
conteo["Medal"] = atletas["Medal"].isna().sum()
faltantes = pd.DataFrame({"faltantes": conteo, "porcentaje": (100 * conteo / len(df)).round(2)})
faltantes = faltantes[faltantes["faltantes"] > 0].sort_values("faltantes", ascending=False)
faltantes.to_csv(f"{TABLAS}/datos_faltantes.csv")
faltantes

,faltantes,porcentaje
notes,266077,98.14
Medal,231333,85.33
IMC,64263,23.70
Weight,62875,23.19
Height,60171,22.19
Age,9474,3.49
region,21,0.01


In [21]:
# Proporción de faltantes antes y después de 1960
df["epoca"] = np.where(df["Year"] < 1960, "Antes de 1960", "1960 en adelante")
falt_epoca = (100 * df.groupby("epoca")[["Age", "Height", "Weight", "IMC", "region"]]
              .agg(lambda s: s.isna().mean())).round(2)
falt_epoca.to_csv(f"{TABLAS}/faltantes_por_epoca.csv")
falt_epoca

,Age,Height,Weight,IMC,region
epoca,,,,,
1960 en adelante,0.53,5.32,5.66,5.92,0.01
Antes de 1960,13.47,79.06,82.28,83.62,0.00


In [22]:
# Porcentaje de faltantes de estatura y peso por década
(100 * df.groupby("decada")[["Age", "Height", "Weight"]].agg(lambda s: s.isna().mean())).round(1)

,Age,Height,Weight
decada,,,
1890,42.9,87.9,87.1
1900,30.4,86.9,88.7
1910,3.9,82.2,85.2
1920,19.0,82.5,87.9
1930,5.1,77.6,85.7
1940,15.7,84.4,84.6
1950,5.8,69.1,69.2
1960,1.4,6.1,6.6
1970,0.7,5.2,5.8


**¿Qué variables tienen faltantes y qué significan?**

| Variable | % faltante | Significado |
|---|---|---|
| Medal | 85,3 % | No es un faltante real: significa que **no ganó medalla** |
| notes | 98,1 % | Solo se escribe cuando el NOC necesita una aclaración |
| Weight / Height | 23,2 % / 22,2 % | No se registró la medida, sobre todo en los primeros Juegos |
| IMC | 23,7 % | Hereda los faltantes de estatura **o** peso |
| Age | 3,5 % | Edad desconocida, casi siempre en atletas muy antiguos |
| region | ~0,01 % | Equipos sin país (refugiados, Tuvalu, desconocido) |

**¿Cambia según la época?** Sí, y muchísimo. Antes de 1960 falta la estatura en el **79,1 %** de los registros y el peso en el **82,3 %**; desde 1960 en adelante solo falta en el **5,3 %** y el **5,7 %**. La edad pasa de **13,5 %** de faltantes a **0,5 %**. Esto muestra que los faltantes **no son aleatorios**: dependen del año (los registros históricos no guardaban esas medidas), así que un análisis de estatura, peso o IMC representa sobre todo a los atletas modernos. Llama la atención un pequeño repunte en los 90 (14 %), que probablemente se debe a fuentes incompletas de esa década.